# PC Controller (P+LPF)

Cheat sheet for some of the parameters symbols
α  β  ρ  θ  ξ  δ
₀ ₁ ₂ ₃ ₄ ₅ ₆ ₇ ₈ ₉
**Transfer function:**  G_c(s) = α₁/(s+1)

**Key properties:**
- **Proportional action only** — always has non-zero steady-state error
- e_ss = r / (1 + G)  where  G = α₁·θ₁·θ₂
- K_P = α₁

**Stability boundary (Routh–Hurwitz, 3rd-order):** G = α₁·θ₁·θ₂ < **8**

## Setup

In [30]:
# while building the package so it refreshes modules 
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [31]:

# Imports
%matplotlib inline
import numpy as np
from bokeh.io import output_notebook, show
import pandas as pd
import sympy as sp
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

# Biomolecular controllers
import biomolecular_controllers as bc
from biomolecular_controllers.gain import pc_stability_boundary


print("✓ Imports successful!")
from biomolecular_controllers.figure_saver import save_fig
from biomolecular_controllers.figure_gifs import save_gif


output_notebook()


✓ Imports successful!


Loading BokehJS ...

In [32]:
output_notebook()

# Initialize tools
runner = bc.SimulationRunner()
calc = bc.MetricsCalculator()
stab = bc.StabilityAnalyzer()
plotter = bc.VisualizationPipeline()
swiffer = bc.SensitivityAnalyzer()


print("✓ Tools initialized!")

Loading BokehJS ...

✓ Tools initialized!


## Parameter Sweep

In [33]:
# Set fixed params such that contribution to any gain(s) not being investigated is ~0.1 or less
# Effective Gain K_p = alpha_1
# Stability: alpha_1*theta_1*theta_2 < 8

alpha_1, theta_1, theta_2 = sp.symbols('alpha_1 theta_1 theta_2', positive=True)

K1 = alpha_1 * theta_1 * theta_2

# sole stability condition from asymptotic analysis, must be > 0
rh_conditions = {
    'P_LPF': 8 - K1   
}

base_params = {
    theta_1: 1.0,   
    theta_2: 1.0,   # theta_1*theta_2 = 1 so KP = alpha_1 directly
    alpha_1: 1.0,   # initialization, overridden when sweeping
}

# example with simulation parameters
# sweep alpha_1, fix everything else
lower, upper = swiffer.get_feasible_range(rh_conditions, alpha_1, base_params)

inside, out_low, out_high, combined = swiffer.build_sweep(lower, upper)
print(f"Feasible range for alpha_1 with small excursion above/below stable regime: ({lower:.4f}, {upper:.4f})")
print(f"Inside:  {np.round(inside, 3)}")
print(f"Below:   {np.round(out_low, 3)}")
print(f"Above:   {np.round(out_high, 3)}")


  P_LPF: upper bound at 8.0000
Feasible range for alpha_1 with small excursion above/below stable regime: (0.0000, 8.0000)
Inside:  [0.4   1.429 2.457 3.486 4.514 5.543 6.571 7.6  ]
Below:   []
Above:   [9.2]


## Part 1: Sweep α₁ (Vary Gain)

G = α₁·θ₁·θ₂.  Stability requires G < 8.

**Steady-state error:** e_ss = r / (1 + G)  — decreases as gain increases but never reaches zero.
The tradeoff is SS error vs overshoot/settling time as α₁ increases toward G_crit = 8.

In [34]:
# Sweep alpha_1 to vary gain G = alpha_1 * theta_1 * theta_2
alpha_vals = np.array(combined)

# Step input: go from 10.0 to 35.0 at t=150
ref_low = bc.DEFAULT_PARAMS["PC"]["ref"]
ref_high = 50.0
t_step = 150.0
t_span = (0, 500)

print(f"Sweeping α₁: {alpha_vals.min():.1f} to {alpha_vals.max():.1f}")
print(f"Step input: {ref_low} → {ref_high} at t={t_step}")

Sweeping α₁: 0.4 to 9.2
Step input: 10.0 → 50.0 at t=150.0


In [35]:
trajectories_dict   = {}
gains_list          = []
ss_errors_list      = []
ss_stds_list  = []
settling_times_list = []
overshoots_list     = []
rise_times_list     = []

params = {
    'theta_1': 1.0,
    'theta_2': 1.0,
    'alpha_1': 1.0,
    'g': 1e6,
}
print('Running PC simulations...')

alpha_vals = np.linspace(0.0, 5.0, 8)

for alpha in alpha_vals:
    current_params = {**params, 'alpha_1': alpha}
    perturbations = [
        {'time': t_step, 'type': 'parameter', 'param': 'ref', 'value': ref_high}
    ]
    try:
        result = runner.run_with_perturbations(
            model='PC', t_span=t_span, points=1200,
            perturbations=perturbations, params=current_params
        )
        time = np.asarray(result['time'],        dtype=float)
        y    = np.asarray(result['states']['y'], dtype=float)
        ref  = np.where(time < t_step, ref_low, ref_high)

        # Effective Closed-loop gain: G = α₁·θ₁·θ₂
        K_P_eff_num = bc.gain.gain_pc(current_params)

        # Use actual SS value for rise_time (SS error may be non-zero when β·k ≠ 1)
        y_ss = float(np.mean(y[-20:]))

        os_r = calc.overshoot(time, y, ref=ref_high, pert_start=t_step, pert_end=t_span[1])
        st_r = calc.settling_time(time, y, ref=ref_high, pert_start=t_step, pert_end=t_span[1])
        rt_r = calc.rise_time(time, y, ref=y_ss, pert_start=t_step, pert_end=t_span[1])
        ss_r = calc.steady_state(time,y, pert_start=t_step, pert_end=t_span[1])

        trajectories_dict[alpha] = {'time': time, 'y': y, 'ref': ref}
        gains_list.append(K_P_eff_num)
        ss_errors_list.append(abs(ss_r['ss_value'] - ref_high))
        ss_stds_list.append(ss_r['ss_std'])
        settling_times_list.append(st_r['settling_time'] if st_r['settled'] else np.nan)
        overshoots_list.append(os_r['magnitude'])
        rise_times_list.append(rt_r['rise_time'] if rt_r['valid'] else np.nan)
    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f'  α₁={alpha:.2f}: failed ({str(e)[:50]})')
        gains_list.append(bc.gain.gain_pc(params))
        ss_errors_list.append(np.nan)
        ss_stds_list.append(np.nan)
        settling_times_list.append(np.nan)
        overshoots_list.append(np.nan)
        rise_times_list.append(np.nan)

gains          = np.array(gains_list)
ss_errors      = np.array(ss_errors_list)
ss_stds        = np.array(ss_stds_list)
settling_times = np.array(settling_times_list)
overshoots     = np.array(overshoots_list)
rise_times     = np.array(rise_times_list)

print(f'✓ Completed {np.sum(~np.isnan(ss_errors))}/{len(alpha_vals)} simulations')
print(f'SS errors: {ss_errors}')

Running PC simulations...
✓ Completed 8/8 simulations
SS errors: [50.         29.16666625 20.58823482 15.90908977 12.96296217 10.93749972
  9.45945851  8.33333294]


## Plot 1: Steady-State Error vs Gain

PC has non-zero SS error: **e_ss = r / (1 + G)** where G = α₁·θ₁·θ₂.
Higher gain → smaller SS error, but approaches instability at G = 8.

In [36]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=alpha_vals,
    trajectories=trajectories_dict,
    metric_name="Steady State Error",
    gain_vals=gains,
    metric_vals=ss_errors,
    title="PC: Steady-State Error vs K_P_eff  [e_ss = r/(1+α₁·θ₁·θ₂)]",
    x_label="Closed-loop gain  G = α₁·θ₁·θ₂",
    x_scale="linear",
    y_scale="linear",
    return_sources=True,
)
show(layout)
#save_fig(layout, 'PC', 'steady_state_error', fmt='png', show=True)
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PC', 'steady_state_error')



[0, 1, 2, 3, 4, 5, 6, 7]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pc_steady_state_error.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pc_steady_state_error.gif')

## Plot 2: Settling Time vs Gain

In [37]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=alpha_vals, 
    trajectories=trajectories_dict,
    metric_name="Settling Time",
    gain_vals=gains,
    metric_vals=settling_times,
    title="PC: Settling Time vs K_P_eff  [e_ss = r/(1+α₁·θ₁·θ₂)]",
    x_label="Closed-loop gain  G = α₁·θ₁·θ₂",
    x_scale="linear",
    y_scale="linear",
    return_sources=True,
)
show(layout)
#save_fig(layout, 'PC', 'settling_time', fmt='png', show=True)
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PC', 'settling_time')



[0, 1, 2, 3, 4, 5, 6, 7]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pc_settling_time.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pc_settling_time.gif')

## Plot 3: Overshoot vs Gain

In [38]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=alpha_vals,
    trajectories=trajectories_dict,
    metric_name="Overshoot",
    gain_vals=gains,
    metric_vals=overshoots,
    title="PC: Overshoot vs K_P_eff  [e_ss = r/(1+α₁·θ₁·θ₂)]",
    x_label="Closed-loop gain  G = α₁·θ₁·θ₂",
    x_scale="linear",
    y_scale="linear",
    return_sources=True,
)
show(layout)
#save_fig(layout, 'PC', 'overshoot', fmt='png', show=True)
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PC', 'overshoot')



[0, 1, 2, 3, 4, 5, 6, 7]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pc_overshoot.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pc_overshoot.gif')

## Plot 4: Rise Time vs Gain

In [39]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=alpha_vals,
    trajectories=trajectories_dict,
    metric_name="Rise Time",
    gain_vals=gains,
    metric_vals=rise_times,
    title="PC: Rise Time vs K_P_eff  [e_ss = r/(1+α₁·θ₁·θ₂)]",
    x_label="Closed-loop gain  G = α₁·θ₁·θ₂",
    x_scale="linear",
    y_scale="linear",
    return_sources=True,
)
show(layout)
#save_fig(layout, 'PC', 'rise_time', fmt='png', show=True)
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PC', 'rise_time')



[0, 1, 2, 3, 4, 5, 6, 7]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pc_rise_time.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pc_rise_time.gif')

## Part 2: Stability Diagram (α₁ vs θ₁)

Boundary: θ₁ = 8 / (α₁·θ₂)  →  G = α₁·θ₁·θ₂ = 8.

### Phase Diagram 1: α₁ vs θ₁

In [40]:
# PC Parameter stability diagram
alpha_range = np.linspace(0.1, 10, 400)
theta_range = np.linspace(0.1, 10, 400)
theta_2_fixed = 1.0

boundary_fn = bc.gain.pc_stability_boundary(theta_2=theta_2_fixed)

p = plotter.plot_stability_diagram(
    x_vals=alpha_range,
    y_vals=theta_range,
    stable_condition=lambda A, T: (A * T * theta_2_fixed < 8),
    boundary_fns=[
        {
            'fn':    boundary_fn,
            'label': 'PC boundary  (G=8)',
            'color': 'black',
            'dash':  'dashed',
            'line_width': 2.0,
        },
    ],
    x_name='α₁',
    y_name='θ₁',
    title=f'PC Parameter Stability Diagram  (θ₂={theta_2_fixed})',
)
p.scatter([-999], [-999], marker="square", size=14,
          fill_color="#8ECF8E", fill_alpha=0.65,
          line_color=None, legend_label="Stable")
p.scatter([-999], [-999], marker="square", size=14,
          fill_color="#C8C8C8", fill_alpha=0.65,
          line_color=None, legend_label="Unstable")

save_fig(p, 'PC', 'stability_diagram', fmt='png', show=True)

  Saved PNG  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pc_stability_diagram.png


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pc_stability_diagram.png')

## Part 3: Root Locus (Dominant Eigenvalue Pair)
Root Locus method: a graphical technique used in control systems to analyze system stability as a gain parameter varies

In [41]:
root_alpha_vals = np.linspace(0.1, 5.5, 12)

eigenvalue_paths = {}
reals, imags, gs = [], [], []
for val in root_alpha_vals:
    p_params = {
        'alpha_1': val,
        'alpha_2': 2.0,
        'theta_1': 1.0,
        'theta_2': 1.0,
        'beta':    1.0,
        'k':       1.001,
    }
    try:
        res  = stab.analyze_stability(model='PC', params=p_params, steady_state_time=100.0)
        eigs = res.eigenvalues
        dom  = eigs[np.argmax(np.real(eigs))]
        reals.append(float(np.real(dom)))
        imags.append(float(np.imag(dom)))
        gs.append(res.gain)
    except Exception:
        reals.append(np.nan); imags.append(np.nan); gs.append(np.nan)
eigenvalue_paths['PC'] = {
    'real': np.array(reals),
    'imag': np.array(imags),
    'gain': np.array(gs),
}

fig = plotter.plot_root_locus(
eigenvalue_paths,
title='P Controllers Root Locus: Dominant Eigenvalue as α₁ varies'
)
save_fig(fig, 'PC', 'root_locus', fmt='png', show=True)

ValueError: zero-size array to reduction operation minimum which has no identity

In [ ]:
# Plot root locus
eigenvalue_paths = {
    'Dominant eigenvalue': {
        'real': eigenvalue_real,
        'imag': eigenvalue_imag
    }
}

fig = plotter.plot_root_locus(
    root_locus_vals,
    eigenvalue_paths,
    title='Root Locus: Dominant Eigenvalue as α₁ varies'
)
show(fig)

TypeError: VisualizationPipeline.plot_root_locus() got multiple values for argument 'title'

In [ ]:
params_to_sweep = {
    'α₁ sweep':  'alpha_1',
    'θ₁ sweep':  'theta_1',
    'θ₂ sweep':  'theta_2',
}
sweep_vals = np.linspace(0.1, 10, 8)

eigenvalue_paths = {}

for label, param_name in params_to_sweep.items():
    reals, imags, gains = [], [], []
    for val in sweep_vals:
        try:
            res = stab.analyze_stability(model="PC", params={param_name: val},
                                        steady_state_time=100.0)
            eigs = res.eigenvalues
            dom = eigs[np.argmax(np.real(eigs))]
            reals.append(np.real(dom))
            imags.append(np.imag(dom))
            gains.append(res.gain)
        except Exception:
            reals.append(np.nan)
            imags.append(np.nan)
            gains.append(np.nan)

    eigenvalue_paths[label] = {
        'real': np.array(reals),
        'imag': np.array(imags),
        'gain': np.array(gains),
    }

fig = plotter.plot_root_locus(
    eigenvalue_paths,
    title='Root Locus: Multi-parameter sweep'
)
show(fig)

ValueError: zero-size array to reduction operation minimum which has no identity

## Create Dashboard

In [ ]:
'''
p1 = viz.plot_stability_phase_diagram(...)
p2 = viz.plot_root_locus(...)
dashboard = viz.create_dashboard([p1, p2], ncols=2)
show(dashboard)
'''